In [ ]:
import json
import re
from typing import List, Dict


units = {
    "शून्य": 0, "एक": 1, "दो": 2, "तीन": 3, "चार": 4,
    "पांच": 5, "छह": 6, "सात": 7, "आठ": 8, "नौ": 9
}

teens = {
    "ग्यारह": 11, "बारह": 12, "तेरह": 13, "चौदह": 14,
    "पंद्रह": 15, "सोलह": 16, "सत्रह": 17, "अठारह": 18, "उन्नीस": 19
}

tens = {
    "दस": 10, "बीस": 20, "तीस": 30, "चालीस": 40,
    "पचास": 50, "साठ": 60, "सत्तर": 70,
    "अस्सी": 80, "नब्बे": 90
}

multipliers = {
    "सौ": 100,
    "हज़ार": 1000,
    "लाख": 100000
}

compound_numbers = {
    "पच्चीस": 25, "छब्बीस": 26, "सत्ताईस": 27,
    "अट्ठाईस": 28, "उनतीस": 29,
    "इकतीस": 31, "बत्तीस": 32, "तैंतीस": 33, "चौंतीस": 34
}

all_numbers = set(
    list(units.keys()) +
    list(teens.keys()) +
    list(tens.keys()) +
    list(multipliers.keys()) +
    list(compound_numbers.keys())
)

idioms = {"दो चार", "दो-चार", "दो तीन", "दो-तीन"}

def is_idiom(text: str) -> bool:
    return text.strip() in idioms

def hindi_words_to_number(tokens: List[str]) -> int:
    total = 0
    current = 0

    for word in tokens:
        if word in units:
            current += units[word]
        elif word in teens:
            current += teens[word]
        elif word in tens:
            current += tens[word]
        elif word in compound_numbers:
            current += compound_numbers[word]
        elif word in multipliers:
            current = max(1, current) * multipliers[word]
            total += current
            current = 0

    return total + current


def normalize_numbers(text: str) -> str:
    if not text:
        return ""

    if is_idiom(text):
        return text

    words = text.split()
    result = []
    buffer = []

    for word in words:
        if word in all_numbers:
            buffer.append(word)
        else:
            if buffer:
                result.append(str(hindi_words_to_number(buffer)))
                buffer = []
            result.append(word)

    if buffer:
        result.append(str(hindi_words_to_number(buffer)))

    return " ".join(result)



english_words = {
    "इंटरव्यू", "जॉब", "प्रॉब्लम", "कंप्यूटर", "मोबाइल",
    "नेटवर्क", "सिस्टम", "फाइल", "कोड", "डेटा", "सॉफ्टवेयर"
}

def is_english_word(word: str) -> bool:
    if word in english_words:
        return True
    return word.endswith(("शन", "मेंट", "टी"))


def tag_english_words(text: str) -> str:
    if not text:
        return ""

    words = text.split()
    tagged = []

    for word in words:
        if is_english_word(word):
            tagged.append(f"[EN]{word}[/EN]")
        else:
            tagged.append(word)

    return " ".join(tagged)


def clean_text(text: str) -> str:
    if not text:
        return ""
    text = re.sub(r"\s+", " ", text)
    return text.strip()



def process_transcript(text: str) -> Dict:
    cleaned = clean_text(text)
    number_normalized = normalize_numbers(cleaned)
    final_output = tag_english_words(number_normalized)

    return {
        "original": text,
        "cleaned": cleaned,
        "number_normalized": number_normalized,
        "final_output": final_output
    }


def load_dataset(json_path: str):
    with open(json_path, "r", encoding="utf-8") as f:
        return json.load(f)


def process_dataset(data):
    results = []

    for item in data:
        asr_text = item.get("asr_output", "")
        reference = item.get("reference", "")

        processed = process_transcript(asr_text)

        results.append({
            "reference": reference,
            **processed
        })

    return results


def save_results(results, output_file="final_output.json"):
    with open(output_file, "w", encoding="utf-8") as f:
        json.dump(results, f, ensure_ascii=False, indent=2)


if __name__ == "__main__":
    input_file = "dataset.json"
    output_file = "final_output.json"

    data = load_dataset(input_file)
    processed_results = process_dataset(data)
    save_results(processed_results, output_file)

    print(f"✅ Processing complete. Output saved to {output_file}")

Processing complete. Output saved to final_output.json
